# Narrating Articles with Venice Text-to-Speech

Turn any web article into a single narrated audio file, and play it back here.

This notebook accompanies [Narrating Articles with Text-to-Speech](https://docs.venice.ai/guides/media/article-narration), which explains the reasoning behind each step. Run the cells in order.

You need a Venice API key from [venice.ai/settings/api](https://docs.venice.ai/guides/getting-started/generating-api-key). A full run scrapes one page, makes one chat completion, and synthesizes about six minutes of speech, so it consumes a small amount of credit.

## Setup

Store the key with the key icon in the Colab sidebar, as a secret named `VENICE_API_KEY`, so it is not saved into the notebook when you share it. If no secret is set you will be prompted for it, and the value stays in memory.

In [ ]:
%pip install -q requests

import os


def load_api_key() -> str:
    try:
        from google.colab import userdata

        return userdata.get('VENICE_API_KEY')
    except Exception:
        pass
    if os.environ.get('VENICE_API_KEY'):
        return os.environ['VENICE_API_KEY']
    from getpass import getpass

    return getpass('Venice API key: ')


# The tutorial code reads the key from the environment.
os.environ['VENICE_API_KEY'] = load_api_key()
print('Key loaded.')

## 1. Choose a model and a voice

Voices belong to models, and sending a voice from one family to a model from another is the most common first mistake. `model_spec.voices` is the authoritative voice list for a model, and `supported_formats` tells you which `response_format` values it accepts.

We use `tts-xai-v1` with the voice `eve`. It supports `pcm`, which is what makes joining chunks straightforward in section 4.

In [ ]:
import requests

models = requests.get(
    'https://api.venice.ai/api/v1/models',
    headers={'Authorization': f"Bearer {os.environ['VENICE_API_KEY']}"},
    params={'type': 'tts'},
    timeout=60,
).json()['data']

for model in models:
    spec = model['model_spec']
    print(f"{model['id']:28} formats={spec['supported_formats']} "
          f"voices={len(spec['voices'])}")

## 2. Make a single request

The response body is raw audio rather than JSON, so write the bytes straight to a file.

In [ ]:
import os
from pathlib import Path

import requests

response = requests.post(
    "https://api.venice.ai/api/v1/audio/speech",
    headers={
        "Authorization": f"Bearer {os.environ['VENICE_API_KEY']}",
        "Content-Type": "application/json",
    },
    json={
        "model": "tts-xai-v1",
        "voice": "eve",
        "input": "Hello from Venice.",
        "response_format": "mp3",
    },
    timeout=300,
)

response.raise_for_status()
Path("hello.mp3").write_bytes(response.content)

In [ ]:
from IPython.display import Audio

Audio('hello.mp3')

## 3. Split text at the 4096 character limit

The `input` field accepts at most 4096 characters, and longer text is rejected outright rather than truncated silently. Splitting on sentence boundaries matters, because a chunk that ends mid sentence produces an audible stumble at the join.

The default `max_chars` is 1500 rather than something near the ceiling, and that is deliberate. Synthesis time grows with input length, so smaller chunks come back sooner and, because they run in parallel, finish the whole job faster.

In [ ]:
from __future__ import annotations

import os
import re
import sys
import wave
from concurrent.futures import ThreadPoolExecutor

import requests

BASE_URL = "https://api.venice.ai/api/v1"
HEADERS = {
    "Authorization": f"Bearer {os.environ['VENICE_API_KEY']}",
    "Content-Type": "application/json",
}

MODEL = "tts-xai-v1"
VOICE = "eve"
SAMPLE_RATE = 24000  # tts-xai-v1 returns 24 kHz mono signed 16-bit PCM.

SENTENCE_END = re.compile(r"(?<=[.!?])\s+")


def split_into_chunks(text: str, max_chars: int = 1500) -> list[str]:
    """Split text on sentence boundaries into chunks under the 4096-character cap."""
    chunks: list[str] = []
    current = ""

    for sentence in SENTENCE_END.split(text.strip()):
        if not sentence:
            continue
        if len(sentence) > max_chars:
            raise ValueError(f"Sentence longer than {max_chars} characters: {sentence[:80]}...")
        if len(current) + len(sentence) + 1 > max_chars:
            chunks.append(current)
            current = sentence
        else:
            current = f"{current} {sentence}" if current else sentence

    if current:
        chunks.append(current)
    return chunks

## 4. Join the chunks into one file

Concatenating encoded audio such as MP3 is unreliable, because every chunk carries its own frame headers. Requesting `pcm` avoids the problem entirely. PCM is raw samples with no container, so joining is just appending bytes, and the standard library `wave` module writes the header for us.

In [ ]:
def synthesize(text: str, speed: float = 1.0) -> bytes:
    """Return raw PCM audio for one chunk."""
    response = requests.post(
        f"{BASE_URL}/audio/speech",
        headers=HEADERS,
        json={
            "model": MODEL,
            "voice": VOICE,
            "input": text,
            "response_format": "pcm",
            "speed": speed,
        },
        timeout=300,
    )
    if response.status_code != 200:
        raise RuntimeError(f"TTS failed ({response.status_code}): {response.text}")
    return response.content


def narrate(text: str, out_path: str) -> str:
    chunks = split_into_chunks(text)
    print(f"Synthesizing {len(chunks)} chunks", file=sys.stderr)

    with ThreadPoolExecutor(max_workers=4) as pool:
        audio = list(pool.map(synthesize, chunks))

    with wave.open(out_path, "wb") as output:
        output.setnchannels(1)
        output.setsampwidth(2)
        output.setframerate(SAMPLE_RATE)
        for part in audio:
            output.writeframes(part)

    seconds = sum(len(part) for part in audio) / 2 / SAMPLE_RATE
    print(f"Wrote {out_path} ({seconds:.1f}s of audio)", file=sys.stderr)
    return out_path

Raw PCM carries no sample rate, so you have to supply the correct one when writing the WAV header, and it is model specific. `tts-xai-v1` returns 24 kHz while `tts-gradium-v1` returns 48 kHz. Guess wrong and the narration plays at the wrong speed and pitch.

To find the rate for any model, ask for one short clip as `wav` and read the header it comes back with.

In [ ]:
import wave

probe = requests.post(
    f'{BASE_URL}/audio/speech',
    headers=HEADERS,
    json={'model': MODEL, 'voice': VOICE, 'input': 'Probe.', 'response_format': 'wav'},
    timeout=300,
)
probe.raise_for_status()
open('probe.wav', 'wb').write(probe.content)

with wave.open('probe.wav') as handle:
    print(handle.getframerate(), handle.getnchannels(), handle.getsampwidth())

## 5. Prepare text that sounds right

Scraped Markdown read aloud verbatim is close to unlistenable. A speech model spells URLs out one character at a time, so `https://docs.venice.ai/llms.txt` comes out as *h t t p s colon slash slash docs dot venice dot a i*. Rather than fighting Markdown with regular expressions, we ask a chat model to rewrite the article as something meant to be spoken.

The page keeps this in a second file that imports `narrate`. Here everything shares one namespace, so that import is dropped.

In [ ]:
from __future__ import annotations

import os
import re
import sys

import requests


BASE_URL = "https://api.venice.ai/api/v1"
HEADERS = {
    "Authorization": f"Bearer {os.environ['VENICE_API_KEY']}",
    "Content-Type": "application/json",
}

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")


def scrape(url: str) -> str:
    response = requests.post(
        f"{BASE_URL}/augment/scrape", headers=HEADERS, json={"url": url}, timeout=120
    )
    response.raise_for_status()
    return response.json()["content"]


def write_script(markdown: str, minutes: int = 6) -> str:
    response = requests.post(
        f"{BASE_URL}/chat/completions",
        headers=HEADERS,
        json={
            "model": "zai-org-glm-5-1",
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "You rewrite articles as scripts to be read aloud. Output plain prose only: "
                        "no Markdown, no headings, no bullet points, no URLs, no code, no emoji. "
                        "Spell out abbreviations and numbers the way a narrator would say them. "
                        "Use short sentences with clear punctuation so speech synthesis paces well."
                    ),
                },
                {
                    "role": "user",
                    "content": f"Rewrite this article as a {minutes}-minute spoken summary.\n\n{markdown[:20000]}",
                },
            ],
            "temperature": 0.4,
        },
        timeout=300,
    )
    response.raise_for_status()
    script = response.json()["choices"][0]["message"]["content"]
    return URL_PATTERN.sub("", script).strip()

## 6. Put it together

The page guards its entry point behind `__main__` and takes the URL from the command line. In the notebook we set it directly, so change `URL` to narrate a different page.

Saving `script.txt` next to the audio is worth the two lines. When a narration sounds wrong the script almost always shows why, and you can fix it without paying to synthesize again.

In [ ]:
URL = 'https://docs.venice.ai/overview/privacy'

print('Scraping', URL)
markdown = scrape(URL)

print(f'Writing script from {len(markdown)} characters of Markdown')
script = write_script(markdown)

with open('script.txt', 'w') as handle:
    handle.write(script)

print(f'Script is {len(script)} characters')
print(script[:400] + '...')

Now that `script` exists, the two inspection cells from the tutorial can run.

In [ ]:
chunks = split_into_chunks(script)
print(f"len(chunks) = {len(chunks)}")
for index, chunk in enumerate(chunks):
    print(f"chunk {index}: {len(chunk)} chars")

In [ ]:
pcm = synthesize(chunks[0])
print(f"bytes   = {len(pcm)}")
print(f"audio   = {len(pcm) / 2 / SAMPLE_RATE:.1f}s")

Synthesize the whole article and listen to it.

In [ ]:
narrate(script, 'article.wav')

Audio('article.wav')

## Streaming for interactive use

Batch narration optimizes total time. A voice interface has the opposite priority, which is getting the first audio out as fast as possible. Setting `streaming: true` returns the body sentence by sentence as it is generated, so playback can start in about a second instead of waiting for the complete clip.

In [ ]:
import time

import requests

start = time.time()
first_byte = None

with requests.post(
    "https://api.venice.ai/api/v1/audio/speech",
    headers=HEADERS,
    json={
        "model": "tts-xai-v1",
        "voice": "eve",
        "input": "Streaming returns audio while the rest is still being generated.",
        "response_format": "mp3",
        "streaming": True,
    },
    stream=True,
    timeout=300,
) as response:
    response.raise_for_status()
    with open("streamed.mp3", "wb") as audio:
        for chunk in response.iter_content(chunk_size=4096):
            if first_byte is None:
                first_byte = time.time() - start
            audio.write(chunk)

print(f"first byte: {first_byte:.2f}s   complete: {time.time() - start:.2f}s")

In [ ]:
Audio('streamed.mp3')

## Next steps

- [Text-to-Speech](https://docs.venice.ai/guides/media/text-to-speech), reference for the endpoint and its parameters
- [Voice Cloning](https://docs.venice.ai/guides/media/voice-cloning), narrate with a custom voice instead of a preset
- [Cited Answers with Web Search](https://docs.venice.ai/guides/tools/cited-web-answers), generate the text this notebook narrates
- [Speech-to-Text](https://docs.venice.ai/guides/media/speech-to-text), transcribe the audio back and compare it against `script.txt` to verify the chunks joined in the right order